In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

print("=" * 80)
print("HW13 – Токенизация, инференс и fine-tuning для классификации эмоций")
print("=" * 80)

# ====================== 1. Seed и устройство ======================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

os.makedirs("artifacts", exist_ok=True)

# ====================== 2. Данные ======================
raw_dataset = load_dataset("emotion")
label_names = raw_dataset["train"].features["label"].names
num_labels = len(label_names)

print(f"Размеры сплитов: train={len(raw_dataset['train'])}, "
      f"validation={len(raw_dataset['validation'])}, test={len(raw_dataset['test'])}")
print(f"Классы: {label_names}\n")

# Примеры
examples = pd.DataFrame({
    "text": [raw_dataset["train"][i]["text"] for i in range(5)],
    "emotion": [label_names[raw_dataset["train"][i]["label"]] for i in range(5)]
})
display(examples)

# ====================== 3. Токенизация ======================
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

print("Токенизация завершена.")

# ====================== 4. Инференс готовой модели ======================
print("\nИнференс готовой модели (SST-2)...")

inference_model_name = "distilbert-base-uncased-finetuned-sst-2-english"
inference_tokenizer = AutoTokenizer.from_pretrained(inference_model_name)
inference_model = AutoModelForSequenceClassification.from_pretrained(inference_model_name).to(device)
inference_model.eval()

inference_texts = [
    "i feel so happy and excited about everything",
    "i am feeling incredibly sad and lonely today",
    "this makes me so angry i cannot stand it",
    "i love her so much it hurts",
    "i am afraid of what might happen next",
    "i cannot believe this happened what a surprise",
]

results = []
for text in inference_texts:
    inputs = inference_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = inference_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0, pred_id].item()
    results.append({
        "text": text,
        "model_label": inference_model.config.id2label[pred_id],
        "confidence": round(confidence, 3)
    })

display(pd.DataFrame(results))

print("\nВывод: готовая SST-2 модель различает только позитив/негатив, а не 6 эмоций → нужен fine-tuning.\n")

# ====================== 5. Fine-tuning ======================
print("Запуск fine-tuning...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
    ignore_mismatched_sizes=True,
).to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./tmp_trainer",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="no",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Удаляем временную папку
if os.path.exists("./tmp_trainer"):
    shutil.rmtree("./tmp_trainer", ignore_errors=True)
    print("Временная папка ./tmp_trainer удалена.")

# ====================== 6. Оценка на test ======================
print("\nФинальная оценка на test...")

test_output = trainer.predict(tokenized_datasets["test"])
test_preds = np.argmax(test_output.predictions, axis=-1)
test_true = test_output.label_ids

test_acc = accuracy_score(test_true, test_preds)
test_f1 = f1_score(test_true, test_preds, average="macro")

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test F1_macro : {test_f1:.4f}")

# ====================== Артефакты ======================
test_texts = raw_dataset["test"]["text"]

sample_df = pd.DataFrame({
    "text": test_texts[:100],
    "true_label": [label_names[l] for l in test_true[:100]],
    "pred_label": [label_names[p] for p in test_preds[:100]],
    "confidence": [round(float(np.max(torch.softmax(torch.tensor(test_output.predictions[i]), dim=-1).numpy())), 4)
                   for i in range(100)]
})
sample_df.to_csv("artifacts/sample_predictions.csv", index=False)

cm = confusion_matrix(test_true, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=200, bbox_inches='tight')
plt.close()

print("\nАртефакты сохранены успешно.")
print("HW13 выполнен.")

HW13 – Токенизация, инференс и fine-tuning для классификации эмоций
Device: cpu



Размеры сплитов: train=16000, validation=2000, test=2000
Классы: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']



,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


Токенизация завершена.

Инференс готовой модели (SST-2)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.190822,0.927000,0.898949
2,No log,0.166574,0.936000,0.907016
3,No log,0.154685,0.941000,0.917990


Временная папка ./tmp_trainer удалена.

Финальная оценка на test...


Test Accuracy : 0.9305
Test F1_macro : 0.8857

Артефакты сохранены успешно.
HW13 выполнен.
